In [5]:
import sys
sys.path.append(r'C:\Users\adeeb\OneDrive\Desktop\olist-ecommerce-analysis\src')

import sqlite3
import pandas as pd
from config import DB_PATH

conn = sqlite3.connect(DB_PATH)
pd.set_option('display.max_columns', None)

In [6]:
sql_path = r'C:\Users\adeeb\OneDrive\Desktop\olist-ecommerce-analysis\sql\01_monthly_revenue.sql'
monthly = pd.read_sql(open(sql_path).read(), conn)
monthly

,year_month,orders,revenue,prev_revenue,mom_growth_pct
0,2016-09,2,207.86,NaN,NaN
1,2016-10,290,44507.30,207.86,21312.2
2,2016-12,1,10.90,44507.30,-100.0
3,2017-01,787,120098.27,10.90,1101719.0
4,2017-02,1718,244959.35,120098.27,104.0
5,2017-03,2617,368341.32,244959.35,50.4
6,2017-04,2377,353842.98,368341.32,-3.9
7,2017-05,3640,503159.19,353842.98,42.2
8,2017-06,3205,429916.61,503159.19,-14.6
9,2017-07,3946,492287.30,429916.61,14.5


### Findings
- **Fast growth through 2017.** Orders rose from 787 in January 2017 to 7,421 in November, about 9.4 times higher. Revenue rose from R$120k to R$1.00M, about 8.4 times. Early growth rates such as +104% in February are inflated by the small starting base.
- **November 2017 was the peak.** Revenue was R$1.00M, up 52.1% on October, then fell 26.1% in December. This may be a Black Friday effect (hypothesis, to be checked with daily order counts).
- **2018 plateaued.** From January to August 2018, revenue stayed between R$838k and R$994k a month, and growth hovered near zero (+1.3% in April, -0.1% in May). Monthly orders fell from 7,168 in March to 6,421 in August, about 10% lower.
- **Data limits.** The analysis covers January 2017 to August 2018. Earlier months have 1 to 290 orders (and November 2016 is missing), and September 2018 has 1 order, so growth rates there are meaningless.

In [14]:
def run_sql(filename):
    path = rf'C:\Users\adeeb\OneDrive\Desktop\olist-ecommerce-analysis\sql\{filename}'
    return pd.read_sql(open(path).read(), conn)

run_sql('02_cumulative_revenue.sql')

,year_month,revenue,cumulative_revenue,moving_avg_3m,pct_of_total
0,2017-01,120098.27,120098.27,120098.27,0.9
1,2017-02,244959.35,365057.62,182528.81,1.8
2,2017-03,368341.32,733398.94,244466.31,2.7
3,2017-04,353842.98,1087241.92,322381.22,2.6
4,2017-05,503159.19,1590401.11,408447.83,3.7
5,2017-06,429916.61,2020317.72,428972.93,3.2
6,2017-07,492287.30,2512605.02,475121.03,3.7
7,2017-08,568245.79,3080850.81,496816.57,4.2
8,2017-09,621415.91,3702266.72,560649.67,4.6
9,2017-10,660179.62,4362446.34,616613.77,4.9


### Findings
- **R$13.45 million in revenue over the 20 months** from January 2017 to August 2018 (excluding canceled and unavailable orders, freight not included).
- **2018 months were much bigger than 2017 months.** The first 8 months of 2018 earned R$7.34M, against R$6.11M for all 12 months of 2017. That is an average month of about R$918k in 2018 versus R$509k in 2017, roughly 1.8 times. The 2017 figure includes the early ramp-up, so the comparison flatters 2018.
- **November 2017 is the biggest single month,** at 7.5% of the 20-month total, against 5% if revenue had been spread evenly.
- **The smoothed trend peaked in May 2018** at about R$989k and eased to about R$863k by August, roughly 13% lower. Three months of softening is worth watching, but it is too short to call a downturn.

In [19]:
cat = run_sql('03_category_revenue.sql')
print("Categories:", len(cat))
print("Total revenue:", round(cat['revenue'].sum(), 2), "(should be 13449529.68, the Task 2 total)")
print("Categories needed to reach 80% of revenue:", (cat['cumulative_pct'] < 80).sum() + 1)
cat.head(10)

Categories: 74
Total revenue: 13449529.68 (should be 13449529.68, the Task 2 total)
Categories needed to reach 80% of revenue: 18


,revenue_rank,category,items_sold,orders,revenue,avg_item_price,pct_of_total,cumulative_pct
0,1,health_beauty,9585,8757,1251145.54,130.53,9.3,9.3
1,2,watches_gifts,5965,5599,1194824.97,200.31,8.9,18.2
2,3,bed_bath_table,11089,9394,1035485.07,93.38,7.7,25.9
3,4,sports_leisure,8573,7656,977728.77,114.05,7.3,33.2
4,5,computers_accessories,7760,6636,902922.70,116.36,6.7,39.9
5,6,furniture_decor,8229,6373,721584.27,87.69,5.4,45.2
6,7,housewares,6903,5838,625538.73,90.62,4.7,49.9
7,8,cool_stuff,3772,3609,619724.39,164.30,4.6,54.5
8,9,auto,4195,3863,585177.48,139.49,4.4,58.8
9,10,garden_tools,4323,3500,479650.06,110.95,3.6,62.4


### Findings
- **Revenue is spread across many categories. The top category, health_beauty, is only 9.3% of - revenue. The top 5 make up 39.9%, the top 10 make up 62.4%, and 18 of the 74 categories are enough to reach 80%.**
- **Volume and revenue rank differently**. bed_bath_table sells the most items (11,089) but ranks third, because its average item price is R$93. watches_gifts sells about half as many items (5,965) but ranks second, at R$200 per item. cool_stuff is similar, with 3,772 items at R$164 each, and it ranks eighth.
- **Sales are driven by everyday home and personal items plus a few pricier categories**The top 10 mix cheap, high-volume categories like bed_bath_table with expensive, low-volume ones like watches_gifts

In [20]:
task4 = run_sql('04_late_delivery_by_state.sql')
direct = pd.read_sql("""
    SELECT COUNT(DISTINCT f.order_id) AS n FROM fact_order_items f
    JOIN dim_date d ON f.purchase_date_key = d.date_key
    WHERE f.order_status = 'delivered' AND f.is_late IS NOT NULL
      AND d.year_month BETWEEN '2017-01' AND '2018-08'""", conn)['n'][0]

print("States:", len(task4))
print("Orders in task4:", task4['delivered_orders'].sum(), "| counted directly:", direct)
task4[task4['delivered_orders'] >= 100].head(10)

States: 27
Orders in task4: 96203 | counted directly: 96203


,state,delivered_orders,late_pct,avg_delivery_days,avg_delay_days
0,AL,397,21.4,24.0,-8.7
1,MA,712,17.6,21.1,-9.4
2,SE,333,15.3,20.9,-9.9
3,PI,475,13.9,19.0,-11.2
4,CE,1271,13.8,20.8,-10.6
6,RJ,12303,12.2,14.8,-11.7
7,BA,3253,12.2,18.9,-10.8
8,PA,942,11.3,23.3,-14.0
9,ES,1991,10.7,15.3,-10.4
10,PB,518,10.4,19.9,-13.2


### Findings
- **Overall late rate: [fill in from the printed line]%.** The state figures below are compared against this.
- **Late deliveries concentrate in the Northeast.** Among states with at least 100 delivered orders, Alagoas has the highest late rate at 21.4%, followed by Maranhão (17.6%), Sergipe (15.3%), Piauí (13.9%) and Ceará (13.8%). These are far from the main seller hub.
- **Rio de Janeiro is the biggest problem by volume.** It has 12,303 delivered orders and a 12.2% late rate, despite a short average delivery of 14.8 days. It is the one to look at first.
- **Paraná is the best large state.** It has 4,907 delivered orders, a 4.1% late rate and an 11.5-day average delivery, the shortest in this view.
- **Late rate measures promises, not speed.** Estimated dates are padded in every state, so orders arrive early on average everywhere. The padding varies: Pará takes 23.3 days and was late 11.3% of the time (14 days early on average), while Amazonas takes 26.0 days but was late only 2.8% of the time (about 20 days early). The remote states with the lowest late rates (Acre, Amapá, Rondônia, Amazonas) are among the slowest, so compare states on delivery days too.
- **Small samples.** Acre (80 orders) and Amapá (67 orders) are below the 100-order threshold, so their 3.8% and 3.0% rest on about 3 and 2 late orders. Don't rank them.

In [26]:
task5 = run_sql('05_seller_segments.sql')
print("Sellers analysed:", len(task5), "of 3095")
print(task5['segment'].value_counts())
print(task5.groupby('segment')['revenue'].sum().round(0))
task5.head(10)

Sellers analysed: 625 of 3095
segment
steady             515
star                73
fix or drop         28
intervene first      9
Name: count, dtype: int64
segment
fix or drop         150697.0
intervene first     194234.0
star               2192889.0
steady             7631793.0
Name: revenue, dtype: float64


,seller_id,state,orders,revenue,late_pct,revenue_tier,reliability,segment
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1124,226987.93,10.5,3,watch,steady
1,53243585a1d6dc2643021fd1853d8905,BA,348,217940.44,3.4,3,reliable,star
2,4a3ca9315b744ce9f8e9374361493884,SP,1772,196882.12,9.7,3,watch,steady
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,578,190917.14,9.2,3,watch,steady
4,7c67e1448b00f6e969d365cea6b010ab,SP,973,186570.05,9.1,3,watch,steady
5,7e93a43ef30c4f03f38b393420bc753a,SP,318,164582.49,4.7,3,reliable,star
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1311,159816.87,6.6,3,watch,steady
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1145,139658.69,5.2,3,watch,steady
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,910,138208.56,7.9,3,watch,steady
9,955fee9216a65b617aa5c0531780ce60,SP,1261,131836.71,6.1,3,watch,steady


In [27]:
overall = (task4['late_pct'] * task4['delivered_orders']).sum() / task4['delivered_orders'].sum()
print("Overall late rate:", round(overall, 1), "%")
print(task5['reliability'].value_counts())
print(task5[task5['segment'] == 'intervene first'][['seller_id', 'state', 'orders', 'revenue', 'late_pct']])

tot = pd.read_sql("""SELECT ROUND(SUM(f.price), 2) AS r FROM fact_order_items f
    JOIN dim_date d ON f.purchase_date_key = d.date_key
    WHERE f.order_status = 'delivered' AND f.is_late IS NOT NULL
      AND d.year_month BETWEEN '2017-01' AND '2018-08'""", conn)['r'][0]
print("Delivered revenue, all sellers:", tot)
print("Share held by the 625 sellers:", round(100 * task5['revenue'].sum() / tot, 1), "%")

Overall late rate: 6.8 %
reliability
watch         304
reliable      284
unreliable     37
Name: count, dtype: int64
                            seller_id state  orders   revenue  late_pct
56   712e6ed8aa4aa1fa65dab41fed5737e4    SC      77  39385.00      20.8
64   06a2c3af7b3aee5d69171b0e14f0ee87    MA     389  36097.98      19.0
75   88460e8ebdecbfecb5f9601833981930    PR     246  31320.40      17.1
130  cd68562d3f44870c08922d380acae552    SP     122  18119.70      15.6
140  abe42c5d03695b4257b5c6cbf4e6784e    RJ      49  17000.00      16.3
176  6039e27294dc75811c0d8a39069f52c0    SP      63  13500.80      17.5
185  0b35c634521043bf4b47e21547b99ab5    PR      50  13235.65      16.0
187  05f51e13da97139648b8125c31e5f51b    MG      57  13100.31      15.8
201  1ca7077d890b907f89be8c954a02686a    SP     108  12474.64      16.7
Delivered revenue, all sellers: 13179777.95
Share held by the 625 sellers: 77.2 %


### Findings
- **Only 625 of 3,095 sellers (20%) are established**, with at least 30 delivered orders in the window. They earned R$10.17M, 77.2% of all delivered revenue (R$13.18M).
- **The overall late rate is 6.8%.** Against that, 284 sellers are reliable (5% or less late), 304 are on watch (5% to 15%) and 37 are unreliable (above 15%).
- **Most established sellers are fine.** 515 are "steady" and 73 are "stars" (top-third revenue, 5% or less late). The 73 stars earn R$2.19M, 22% of established-seller revenue, from 12% of the sellers.
- **"Steady" is not "reliable".** The largest seller (4869f7a5..., R$227k) is classed steady with a 10.5% late rate, about 1.5 times the average.
- **The seller-side lateness problem is small in revenue terms.** The 37 unreliable sellers (6%) carry R$345k, 3.4% of established-seller revenue. The nine "intervene first" sellers are upper-tier by revenue with late rates above 15%, worth R$194k together, so this is a watch list rather than a major risk.
- **Late sellers are not concentrated in one region.** The nine are spread across six states (three in SP, two in PR, and one each in SC, MA, RJ and MG).
- **Two of the ten biggest sellers (4869f7a5... and 955fee92...) are also among the four sellers behind most R$0 freight rows in Phase 1.** Worth investigating.

In [28]:
task6 = run_sql('06_customer_segments.sql')
print("Customers:", task6['customers'].sum())
print("Revenue:", round(task6['revenue'].sum(), 2), "(should be 13449529.68, the Task 2 total)")
task6

Customers: 94703
Revenue: 13449529.68 (should be 13449529.68, the Task 2 total)


,segment,customers,pct_customers,revenue,pct_revenue,avg_recency_days,avg_spend
0,middle,30582,32.3,4069333.12,30.3,223.0,133.06
1,lapsed big spender,9786,10.3,3046373.64,22.7,418.0,311.30
2,recent big spender,9885,10.4,3009851.02,22.4,80.0,304.49
3,recent,20639,21.8,1295673.64,9.6,78.0,62.78
4,lapsed,20937,22.1,1281173.54,9.5,419.0,61.19
5,repeat customer,2874,3.0,747124.72,5.6,224.0,259.96


### Findings
- **94,703 customers generated R$13.45M**, about R$142 per customer. (This is fewer than the 96,096 people in dim_customer because canceled and unavailable orders and months outside the window are excluded.)
- **Almost nobody comes back.** Only 2,874 customers (3.0%) placed more than one order. They produced R$747k, 5.6% of revenue, and spent R$260 on average, 1.8 times the typical customer. Growth has come from new customers, and retention is the biggest gap.
- **Spend is concentrated.** The two big-spender segments (one-time buyers in the top third by spend, at the recency extremes) are 20.8% of customers and 45.0% of revenue, averaging R$305 to R$311 each. The two small-spender segments (recent and lapsed) are 43.9% of customers and 19.2% of revenue, at about R$62 each, about one-fifth as much. The "middle" group also contains some big spenders, so the true concentration is higher.
- **Lapsed means no repurchase in the data, not lost.** "Recent" customers last bought about 80 days before the data ends, "lapsed" customers about 14 months before. The data stops in August 2018, so some lapsed customers may have returned later.
- **Win-back opportunity.** 9,786 lapsed big spenders (average R$311) hold R$3.05M of past revenue, the second-largest segment after "middle".

In [31]:
task7 = run_sql('07_cohort_retention.sql')
print("Cohort sizes add up to:", task7[task7['months_since'] == 0]['active'].sum(),
      "(should be 94703, the Task 6 customer count)")
pivot = task7.pivot(index='cohort', columns='months_since', values='retention_pct')
pivot.iloc[:, :7].round(2)

Cohort sizes add up to: 94703 (should be 94703, the Task 6 customer count)


months_since,0,1,2,3,4,5,6
cohort,,,,,,,
2017-01,100.0,0.40,0.27,0.13,0.40,0.13,0.40
2017-02,100.0,0.24,0.30,0.12,0.41,0.12,0.24
2017-03,100.0,0.51,0.35,0.39,0.35,0.16,0.16
2017-04,100.0,0.60,0.21,0.17,0.30,0.26,0.34
2017-05,100.0,0.48,0.48,0.40,0.31,0.34,0.42
2017-06,100.0,0.45,0.35,0.39,0.26,0.39,0.35
2017-07,100.0,0.52,0.34,0.24,0.29,0.21,0.31
2017-08,100.0,0.68,0.34,0.27,0.36,0.53,0.29
2017-09,100.0,0.69,0.54,0.29,0.44,0.22,0.22


### Findings
- **Almost no customers order again in any given month.** Month-1 retention ranges from 0.22% (December 2017 cohort) to 0.71% (October 2017 cohort), and averages about 0.5% for 2017 cohorts and 0.45% for 2018 cohorts (unweighted averages of the cohort percentages). That is roughly 2 to 7 returning customers per 1,000 acquired.
- **There is no drop-off curve.** Months 2 to 6 sit in the same 0.1% to 0.5% range as month 1, with no second-purchase window. Repeat orders trickle in at any time.
- **Retention has not improved.** The 2018 cohorts look like the 2017 ones, so growth came from acquiring new customers, not from keeping them.
- **The November 2017 cohort is the largest and not stickier.** Its month-1 retention is 0.56%, close to the average. Whether these were Black Friday shoppers is a hypothesis. The December 2017 cohort has the lowest month-1 retention, 0.22%.
- **Consistent with Task 6.** Each month adds only a tiny share, but over up to 19 months it builds to the 3.0% of customers who ordered more than once.
- **Noise warning.** The January 2017 percentages move in steps of 0.13, which implies a cohort of about 750 customers, so one customer shifts the value by 0.13 points. Don't read meaning into small differences between cohorts.
- **Limits.** Empty cells mean "not observable yet", not 0%. Customers who ordered in late 2016 and again in 2017 are counted as new in 2017, which is negligible (about 300 orders in 2016).

In [30]:
task8 = run_sql('08_payment_behaviour.sql')

direct = pd.read_sql("""
    SELECT COUNT(DISTINCT f.order_id) AS orders,
           ROUND(SUM(f.price + f.freight_value), 2) AS items_plus_freight
    FROM fact_order_items f JOIN dim_date d ON f.purchase_date_key = d.date_key
    WHERE f.order_status NOT IN ('canceled', 'unavailable')
      AND d.year_month BETWEEN '2017-01' AND '2018-08'""", conn)

print("Orders in task8:", task8['orders'].sum(), "| orders in scope:", direct['orders'][0])
print("Paid in task8:", round(task8['paid'].sum(), 2), "| items + freight:", direct['items_plus_freight'][0])

by_type = task8.groupby('main_type')[['orders', 'multi_payment_orders', 'paid']].sum()
by_type['pct_orders'] = (100 * by_type['orders'] / by_type['orders'].sum()).round(1)
print(by_type.sort_values('orders', ascending=False))
task8

Orders in task8: 97905 | orders in scope: 97905
Paid in task8: 15686469.07 | items + freight: 15683706.74
             orders  multi_payment_orders         paid  pct_orders
main_type                                                         
credit_card   73888                   990  12308949.97        75.5
boleto        19477                     0   2817399.95        19.9
voucher        3028                  1918    347993.13         3.1
debit_card     1512                     1    212126.02         1.5


,main_type,installment_band,orders,multi_payment_orders,paid
0,credit_card,1 (full payment),23678,246,2387523.78
1,credit_card,2-3,22345,305,3023665.38
2,boleto,1 (full payment),19477,0,2817399.95
3,credit_card,4-6,15919,230,2900255.03
4,credit_card,7-10,11610,197,3875789.20
5,voucher,1 (full payment),3028,1918,347993.13
6,debit_card,1 (full payment),1512,1,212126.02
7,credit_card,11+,336,12,121716.58


### Findings
- **Credit card dominates.** It leads 75.5% of orders (73,888) and 78.5% of money paid (R$12.3M). Boleto leads 19.9% of orders and 18.0% of money, voucher 3.1% and 2.2%, debit card 1.5% and 1.4%. Each order is labelled by its largest payment.
- **Most card customers use installments.** Only 32.0% of credit-card orders are paid in full. 30.2% use 2-3 installments, 21.5% use 4-6, 15.7% use 7-10, and 0.5% use 11 or more.
- **More installments go with bigger baskets.** Average paid per credit-card order rises from R$101 (full payment) to R$135 (2-3), R$182 (4-6), R$334 (7-10) and R$362 (11+). Installments probably make larger purchases affordable, but this data doesn't prove cause.
- **Boleto is always a single payment.** All 19,477 boleto orders fall in the full-payment band, and none has a second payment. The average is R$145.
- **Vouchers are mostly a top-up.** 1,918 of the 3,028 voucher-led orders (63%) also include another payment method. Across all orders, 2,909 (3.0%) have more than one payment row.
- **Freight is a large add-on.** Items plus freight is R$15.68M, and Task 2's item revenue is R$13.45M, so freight is about R$2.23M, 16.6% on top of item prices.
- **Payments exceed items plus freight by R$2,762 (0.018%).** The cause may be installment interest or rounding (hypothesis, not checked).
- **Limit.** Labelling by the largest payment hides small voucher payments next to a larger card payment, so voucher use is understated.

In [32]:
reviews = pd.read_sql("SELECT * FROM order_reviews", conn)
order_ids = pd.read_sql("SELECT order_id FROM orders", conn)['order_id']

print("Rows:", len(reviews))
print("Unique review_id:", reviews['review_id'].nunique())
print("Unique order_id:", reviews['order_id'].nunique())
print("Orders with more than one review:", (reviews.groupby('order_id').size() > 1).sum())
print("Scores outside 1-5:", (~reviews['review_score'].between(1, 5)).sum())
print("Reviews with no matching order:", (~reviews['order_id'].isin(order_ids)).sum())

Rows: 99224
Unique review_id: 98410
Unique order_id: 98673
Orders with more than one review: 547
Scores outside 1-5: 0
Reviews with no matching order: 0


order_reviews has 99,224 rows but only 98,673 unique orders. 547 orders have more than one review, which accounts for the extra 551 rows.
review_id repeats too (99,224 rows, 98,410 unique). We key on order_id, so that doesn't affect us.
There are no scores outside 1 to 5 and no orphan reviews. Since all 98,673 reviewed orders exist in orders, exactly 768 of the 99,441 orders have no review, so Task 9 can only cover reviewed orders.

In [33]:
#Build fact_reviews with one review per order. The rule is to keep the most recent review. A later review reflects how the customer felt in the end, after any problem was resolved. The cell also shows how often the duplicate reviews disagree, so you know whether the rule matters:


reviews = pd.read_sql("SELECT * FROM order_reviews", conn)

multi = reviews[reviews.duplicated('order_id', keep=False)]
spread = multi.groupby('order_id')['review_score'].agg(lambda s: s.max() - s.min())
print("Orders with several reviews:", multi['order_id'].nunique())
print("...whose scores differ:", (spread > 0).sum())

latest = (reviews
    .sort_values(['order_id', 'review_creation_date', 'review_answer_timestamp', 'review_id'])
    .drop_duplicates('order_id', keep='last'))

fact_reviews = pd.DataFrame({
    'order_id': latest['order_id'],
    'review_id': latest['review_id'],
    'review_score': latest['review_score'],
    'has_comment': latest['review_comment_message'].notna().astype(int),
    'review_creation_date': latest['review_creation_date'].str[:10],
})
fact_reviews.to_sql('fact_reviews', conn, if_exists='replace', index=False)

print("Rows in fact_reviews:", len(fact_reviews), "(should be 98673)")
print("Unique orders:", fact_reviews['order_id'].nunique())
fact_reviews['review_score'].value_counts().sort_index()

Orders with several reviews: 547
...whose scores differ: 202
Rows in fact_reviews: 98673 (should be 98673)
Unique orders: 98673


review_score
1    11366
2     3129
3     8133
4    19044
5    57001
Name: count, dtype: int64

In [34]:
task9 = run_sql('09_late_vs_reviews.sql')
print("Orders analysed:", task9['orders'].sum(), "| delivered orders in scope (Task 4): 96203")
task9

Orders analysed: 95560 | delivered orders in scope (Task 4): 96203


,delivery_band,orders,avg_score,pct_1_2_star,pct_5_star
0,1. 8+ days early,70673,4.32,9.0,63.7
1,2. 1-7 days early,17229,4.20,10.2,57.5
2,3. on the estimated day,1280,4.03,12.4,50.6
3,4. 1-3 days late,1851,3.29,32.1,33.2
4,5. 4-7 days late,1748,2.10,67.6,14.1
5,6. 8+ days late,2779,1.70,79.3,7.0


### Findings
- **Late delivery is the clearest driver of bad ratings in this data.** Orders delivered on or before the estimated day average about 4.3 stars, and late orders about 2.3, a gap of roughly two points. The share of 1 to 2 star reviews goes from about 9% to about 62%, nearly seven times higher.
- **The damage grows with the delay.** Compared with orders delivered on the estimated day (4.03 stars, 12.4% 1-2 star), 1 to 3 days late drops to 3.29 stars (32.1%), 4 to 7 days late to 2.10 (67.6%), and 8 or more days late to 1.70 (79.3%).
- **Even a short delay hurts.** Being 1 to 3 days late costs about 0.7 stars, and about one in three of those customers leaves a 1 or 2 star review.
- **Arriving early helps, but only a little.** Scores rise from 4.03 (on the estimated day) to 4.20 (1 to 7 days early) and 4.32 (8 or more days early). The "on the estimated day" band has only 1,280 orders, so treat it as noisy.
- **Most customers get early deliveries.** 74% of analysed orders (70,673) arrived 8 or more days early, because estimated dates are padded (see Task 4). Only 6.7% of orders (6,378) were late, so the rating problem is concentrated in a small share of orders.
- **Association, not proof.** Late orders may also have other problems such as damaged or wrong items, so lateness may not be the only cause (hypothesis). Phase 4 tests this properly.
- **Coverage.** 95,560 delivered orders with a review are analysed (99.3% of 96,203). Orders without a review are excluded, and customers who review may differ from those who don't.